# MAKING DATA STRUCTURES THREAD-SAFE

### 1. Extending Python Data Structures' Class definitions to include locks

Author mentions this is one way of working with data structures while writing thread-safe code. Apparently it's the best, and also the most involved way.

What I do not understand is, if this is a common-enough tactic, why aren't there popular library implementations of the same? Perhaps there are... i'll research once done with this section.

In [ ]:
# lock version of set

from threading import Lock

class LockedSet(set):
    """A set where add(), remove(), and 'in' operator are thread-safe"""
    def __init__(self, *args, **kwargs):
        self._lock = Lock()
        super().__init__(*args, **kwargs) # super means parent

    def add(self, elem):
        with self._lock:
            super().add(elem)

    def remove(self, elem):
        with self._lock:
           super().remove(elem)

    def __contains__(self, elem):
        with self._lock:
            return super().__contains__(elem)

try out others on your own, and play with them!

### 2. Using method decorators

Use them like this...

In [ ]:
def lock(method):
    def newmethod(self, *args, **kwargs):
        with self._lock:
            return method(self, *args, **kwargs)
    return newmethod

class DecoratorLockedSet(set):
    def __init__(self, *args, **kwargs):
        self._lock = Lock()
        super().__init__(*args, **kwargs)

    @lock
    def add(self, elem):
        return super().add(elem)

    @lock
    def remove(self, elem):
        return super().remove(elem)

# not sure how this is any better than just extending the methods themselves...

### 3. Class decorators

Method decorators but taken one step further. Apply it to a class and the decorator applies to all its methods.

In [ ]:
from threading import Lock

def lock_class(methodnames, lockfactory):
    return lambda cls: make_threadsafe(cls, methodnames, lockfactory)

def lock_method(method):
    if getattr(method, '__is_locked', False):
        raise TypeError("Method %r is already locked!" %method)

    def locked_method(self, *arg, **kwarg):
        with self._lock:
            return method(self, *arg, **kwarg)

    locked_method.__name__ = '%s(%s)' % ('lock_method', method.__name__)
    locked_method.__is_locked = True
    return locked_method

def make_threadsafe(cls, methodnames, lockfactory):
    init = cls.__init__
    def newinit(self, *arg, **kwarg):
        init(self, *arg, **kwarg)
        self._lock = lockfactory()
    cls.__init__ = newinit
    for methodname in methodnames:
        oldmethod = getattr(cls, methodname)
        newmethod = lock_method(oldmethod)
        setattr(cls, methodname, newmethod)
    return cls

@lock_class(['add','remove'], Lock)
class ClassDecoratorLockedSet(set):
    @lock_method  # if you double-lock a method, a TypeError is raised
    def lockedMethod(self):
        print("This section of our code would be thread safe")